CI/CD with GitHub Actions

**Module 2 · Python for AI Testing & Automation**

---

## What we'll cover

| # | Topic | Why it matters |
|---|---|---|
| 1 | What CI/CD means in practice | Tests that don't run automatically don't run |
| 2 | GitHub Actions concepts | Workflows, jobs, steps, triggers |
| 3 | The workflow file — dissected | Every line of `llm-tests.yml` explained |
| 4 | Secrets — handling API keys | Never in code, always in GitHub Secrets |
| 5 | Caching pip dependencies | 3x faster CI runs |
| 6 | Artifacts — uploading reports | Making test results accessible |
| 7 | Branch protection — gating merges | The teeth behind the tests |
| 8 | Scheduling — nightly runs | Catching model drift |

---

---
## 1. What CI/CD Means in Practice

**CI (Continuous Integration):** every time code changes, run the tests automatically — on a fresh machine, not your laptop.

**CD (Continuous Delivery/Deployment):** automatically deploy if tests pass.

For AI testing, CI gives us:
- Tests that can't be skipped
- A record of every test run, forever
- Automatic gate: broken evals can't merge
- Scheduled runs to catch model drift (even when no code changes)

> **Plain English:** CI is a tiny person who sits in the cloud, watching your repository. Every time someone pushes code, this person wakes up, checks out the code on a fresh computer, runs every test, and reports the result. You didn't ask them — they just do it, forever.

---
## 2. GitHub Actions Concepts

```
Workflow (YAML file in .github/workflows/)
  └── Triggered by: push, pull_request, schedule, workflow_dispatch
       └── Job (runs on a VM: ubuntu-latest, macos-latest, windows-latest)
            ├── runs-on: ubuntu-latest
            └── Steps (run in sequence on that VM)
                 ├── Step: actions/checkout@v4
                 ├── Step: actions/setup-python@v5
                 ├── Step: pip install -r requirements.txt
                 └── Step: pytest ...
```

**Key vocabulary:**

| Term | Analogy |
|---|---|
| Workflow | A checklist that runs itself |
| Job | One machine with a set of tasks |
| Step | One item on the checklist |
| Runner | The cloud VM that runs the job |
| Trigger | The event that starts the workflow |
| Secret | An encrypted env var the job can access |
| Artifact | A file the job produces and saves |

---
## 3. The Workflow File — Line by Line

Let's read the actual `llm-tests.yml` from the repository and walk through every section.

In [2]:
from pathlib import Path
workflow = Path("github_actions/llm-tests.yml").read_text()
print(workflow)

# Day 7 — GitHub Actions workflow for the AI test suite.
#
# Copy this into `.github/workflows/llm-tests.yml` at the repo root.
# Before merging, configure repo secrets at:
#   Settings → Secrets and variables → Actions → New repository secret
# Add:
#   OPENAI_API_KEY      (required if PROVIDER=openai)
#   ANTHROPIC_API_KEY   (required if PROVIDER=anthropic)

name: LLM Tests

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]
  workflow_dispatch:        # allow manual runs from the Actions tab
  schedule:
    - cron: "0 6 * * 1"     # weekly nightly run, Monday 06:00 UTC

jobs:
  test:
    name: Run pytest suite
    runs-on: ubuntu-latest
    timeout-minutes: 15

    # Reasonable defaults — override per repo as needed
    env:
      PROVIDER: openai
      DEMO_MODEL: gpt-4o-mini
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
      ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}

    steps:
      - name: Checkout
        uses: actions/checkout@v4

      - na

### Section-by-section breakdown

In [1]:
# Read and explain each YAML section
explanations = [
    ("name:", 
     "The display name in the GitHub Actions UI. Pick something descriptive — you'll scan this list."),
    ("on:", 
     "Trigger events. push.branches limits to main. pull_request also runs on PRs. "
     "workflow_dispatch adds a manual 'Run workflow' button in the UI."),
    ("runs-on:", 
     "The VM type. ubuntu-latest is cheapest and fastest. Use macos-latest only if you test "
     "platform-specific behavior."),
    ("env:", 
     "Environment variables available to every step in the job. Secrets are injected here safely — "
     "they never appear in logs."),
    ("actions/checkout@v4", 
     "Check out the repository code into the runner. Always the first step."),
    ("actions/setup-python@v5", 
     "Install Python. pin-version to a specific minor version (3.11) for reproducibility."),
    ("actions/cache@v4", 
     "Cache the pip download cache between runs. Saves 30-60s per run — on a 3-min workflow that's 30%."),
    ("pip install", 
     "Install dependencies. Use --no-deps on repeat runs if you've pinned with pip freeze."),
    ("pytest", 
     "Run your test suite. -v for verbose (shows each test name). --tb=short for readable tracebacks."),
    ("upload-artifact", 
     "Save any file as an artifact — accessible in the Actions UI for 90 days by default. "
     "HTML reports, coverage files, screenshots — anything your CI produces."),
]

for keyword, explanation in explanations:
    print(f"\n{'─'*50}")
    print(f"  ► {keyword}")
    print(f"    {explanation}")


──────────────────────────────────────────────────
  ► name:
    The display name in the GitHub Actions UI. Pick something descriptive — you'll scan this list.

──────────────────────────────────────────────────
  ► on:
    Trigger events. push.branches limits to main. pull_request also runs on PRs. workflow_dispatch adds a manual 'Run workflow' button in the UI.

──────────────────────────────────────────────────
  ► runs-on:
    The VM type. ubuntu-latest is cheapest and fastest. Use macos-latest only if you test platform-specific behavior.

──────────────────────────────────────────────────
  ► env:
    Environment variables available to every step in the job. Secrets are injected here safely — they never appear in logs.

──────────────────────────────────────────────────
  ► actions/checkout@v4
    Check out the repository code into the runner. Always the first step.

──────────────────────────────────────────────────
  ► actions/setup-python@v5
    Install Python. pin-version to 

---
## 4. Secrets — The Non-Negotiable Rule

**Never commit an API key.** Not even "just for testing." Not even "for a few minutes."

GitHub scans all public repos for known secret patterns (OpenAI keys start with `sk-`, Anthropic with `sk-ant-`). You'll get an email. The key is already compromised.

### How GitHub Secrets work

1. Go to your repo → **Settings** → **Secrets and variables** → **Actions**
2. Click **New repository secret**
3. Name: `OPENAI_API_KEY` — Value: your actual key
4. In the workflow YAML: `${{ secrets.OPENAI_API_KEY }}`
5. GitHub injects it as an environment variable — **never visible in logs**

In [4]:
# How secrets appear in workflow YAML (this is a code cell, not actual YAML)
secrets_yaml = """
# In the workflow file:
jobs:
  test:
    runs-on: ubuntu-latest
    env:
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}     # ← GitHub injects this
      ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
      PROVIDER: openai   # non-secret config can be inline

    steps:
      - name: Run tests
        run: pytest tests/ -v
        # The OPENAI_API_KEY env var is now available to pytest
        # and to any Python code it runs
"""
print(secrets_yaml)
print("Rule: ${{ secrets.X }} → safe. Anything hardcoded in the file → never.")


# In the workflow file:
jobs:
  test:
    runs-on: ubuntu-latest
    env:
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}     # ← GitHub injects this
      ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
      PROVIDER: openai   # non-secret config can be inline

    steps:
      - name: Run tests
        run: pytest tests/ -v
        # The OPENAI_API_KEY env var is now available to pytest
        # and to any Python code it runs

Rule: ${{ secrets.X }} → safe. Anything hardcoded in the file → never.


In [5]:
# What GitHub masks in logs
masked_example = """
If your workflow accidentally prints a secret (BAD!), GitHub replaces it with ***.

Example log output:
  echo $OPENAI_API_KEY
  → *** (masked)

But: if you do clever things like base64-encode the key, masking may not catch it.
Bottom line: never print secrets. Use environment variables — never echo them.
"""
print(masked_example)


If your workflow accidentally prints a secret (BAD!), GitHub replaces it with ***.

Example log output:
  echo $OPENAI_API_KEY
  → *** (masked)

But: if you do clever things like base64-encode the key, masking may not catch it.
Bottom line: never print secrets. Use environment variables — never echo them.



---
## 5. Caching pip Dependencies

Pip installs are slow — downloading and installing 20+ packages can take 60-90 seconds. Caching keeps a copy of the download directory so future runs skip the network fetch.

In [ ]:
caching_yaml = """
# Without caching: pip install takes 60-90s every run.
# With caching: 5-15s after the first run.

- name: Cache pip
  uses: actions/cache@v4
  with:
    path: ~/.cache/pip
    # Cache key includes OS + Python version + hash of requirements.txt
    # If requirements.txt changes → cache miss → fresh install
    key: ${{ runner.os }}-pip-${{ hashFiles('**/requirements.txt') }}
    restore-keys: |
      ${{ runner.os }}-pip-

- name: Install dependencies
  run: pip install -r requirements.txt
  # On cache hit: pip sees the packages are already downloaded → skips network
  # On cache miss: normal install, cache is saved for next run
"""
print(caching_yaml)
print("Savings: 60-90s → 5-15s. On a 5-min workflow, that's 20% faster. Free.")

---
## 6. Artifacts — Capturing Test Output

In [ ]:
artifact_yaml = """
# After pytest runs, upload the HTML report as an artifact.
# It's accessible from the GitHub Actions UI for 90 days.

- name: Run tests
  run: |
    cd module-02-python-for-ai-testing/examples/test_framework
    pytest -v --html=report.html --self-contained-html
  continue-on-error: true   # Upload artifact even if tests fail

- name: Upload test report
  uses: actions/upload-artifact@v4
  if: always()              # Always upload, even on failure
  with:
    name: pytest-report-${{ github.run_number }}
    path: "**/report.html"
    retention-days: 30      # Default is 90 days

# After the run, go to:
# Actions tab → [your workflow run] → Artifacts section → Download
"""
print(artifact_yaml)

---
## 7. Branch Protection — The Teeth

Tests without enforcement are suggestions. Branch protection turns them into gates.

### Setup:
1. GitHub repo → **Settings** → **Branches** → **Add rule**
2. Branch name pattern: `main`
3. Check: **Require status checks to pass before merging**
4. Search for your workflow job name (e.g., `test`)
5. Check: **Require branches to be up to date before merging**
6. Save

**Result:** if any test in your CI workflow fails, the PR cannot be merged — even if the reviewer approves it. AI regressions can't ship without someone fixing them first.

In [ ]:
print("""
Branch Protection Flow:

  Developer pushes a prompt change
         ↓
  GitHub Actions runs the test suite
         ↓
  If tests pass: ✓ green check → merge allowed
  If tests fail: ✗ red X → merge BLOCKED
         ↓
  Developer must fix the regression
  before anyone can merge to main

This is how AI testing gets teeth.
""")

---
## 8. Scheduled Runs — Catching Model Drift

Model providers update models without notice. A scheduled eval catches silent regressions even when you haven't pushed any code.

In [6]:
schedule_yaml = """
# Add to the 'on:' section to run at a schedule:
on:
  push:
    branches: [main]
  pull_request:
    branches: [main]
  schedule:
    - cron: '0 6 * * 1-5'   # 6:00 UTC Mon-Fri (weekdays only)
    # Cron format: minute hour day-of-month month day-of-week
    # '0 6 * * 1-5' = 6am UTC, Monday through Friday
    # '0 0 * * 0'   = midnight UTC, every Sunday

# Example nightly drift check:
nightly-drift:
  name: Nightly Model Drift Check
  runs-on: ubuntu-latest
  # Only runs on schedule — not on push/PR
  if: github.event_name == 'schedule'
  steps:
    - uses: actions/checkout@v4
    - uses: actions/setup-python@v5
      with: {python-version: '3.11'}
    - run: pip install -r requirements.txt
    - run: pytest tests/ -v --html=drift-report.html
      env:
        OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
    - uses: actions/upload-artifact@v4
      if: always()
      with:
        name: drift-report-${{ github.run_id }}
        path: drift-report.html
"""
print(schedule_yaml)


# Add to the 'on:' section to run at a schedule:
on:
  push:
    branches: [main]
  pull_request:
    branches: [main]
  schedule:
    - cron: '0 6 * * 1-5'   # 6:00 UTC Mon-Fri (weekdays only)
    # Cron format: minute hour day-of-month month day-of-week
    # '0 6 * * 1-5' = 6am UTC, Monday through Friday
    # '0 0 * * 0'   = midnight UTC, every Sunday

# Example nightly drift check:
nightly-drift:
  name: Nightly Model Drift Check
  runs-on: ubuntu-latest
  # Only runs on schedule — not on push/PR
  if: github.event_name == 'schedule'
  steps:
    - uses: actions/checkout@v4
    - uses: actions/setup-python@v5
      with: {python-version: '3.11'}
    - run: pip install -r requirements.txt
    - run: pytest tests/ -v --html=drift-report.html
      env:
        OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
    - uses: actions/upload-artifact@v4
      if: always()
      with:
        name: drift-report-${{ github.run_id }}
        path: drift-report.html



---
## 9. Pushing the Workflow to Your Repo

In [1]:
deployment_steps = """
Steps to activate CI on your repo:

1. Create the directory:
   mkdir -p .github/workflows

2. Copy the workflow file:
   cp module-02-python-for-ai-testing/examples/github_actions/llm-tests.yml \
      .github/workflows/llm-tests.yml

3. Add your API key as a GitHub Secret:
   GitHub repo → Settings → Secrets and variables → Actions → New repository secret
   Name: OPENAI_API_KEY
   Value: sk-...

4. Commit and push:
   git add .github/workflows/llm-tests.yml
   git commit -m "ci: add LLM test workflow"
   git push

5. Go to the Actions tab in your repo and watch it run.
"""
print(deployment_steps)


Steps to activate CI on your repo:

1. Create the directory:
   mkdir -p .github/workflows

2. Copy the workflow file:
   cp module-02-python-for-ai-testing/examples/github_actions/llm-tests.yml       .github/workflows/llm-tests.yml

3. Add your API key as a GitHub Secret:
   GitHub repo → Settings → Secrets and variables → Actions → New repository secret
   Name: OPENAI_API_KEY
   Value: sk-...

4. Commit and push:
   git add .github/workflows/llm-tests.yml
   git commit -m "ci: add LLM test workflow"
   git push

5. Go to the Actions tab in your repo and watch it run.



---
## Module 2 Summary

### What you built over 7 sessions:

| Day | Output |
|---|---|
| Day 1 | A working Python environment + syntax fluency |
| Day 2 | JSON parsing + golden dataset format |
| Day 3 | Robust LLM client with retries + structured logging |
| Day 4 | Multi-provider LLMClient + normalized response |
| Day 5 | pytest suite: fixtures, parametrize, markers |
| Day 6 | Framework: conftest, golden data, parallel, HTML reports |
| Day 7 | Green CI pipeline gating merges on test results |

**This stack:**
- Scales from 10 to 10,000 test cases without structural changes
- Runs on every commit, automatically
- Cannot be bypassed (branch protection)
- Detects model drift on a schedule
- Produces shareable HTML reports

This is the foundation everything in Modules 3–9 builds on.

**Exercise:** [`exercises/github_actions_exercise.md`](../exercises/github_actions_exercise.md)  
**Next Module:** Module 3 — AI Testing Fundamentals: what you're actually testing for, and why
